# Model Evaluation - Manufacturing Predictive Maintenance

Confusion matrix, precision/recall, and feature importance for the trained RandomForest model.

Reads from: gold_ml_features and the saved model at Files/models/maintenance_failure_rf
Writes to: gold_ml_feature_importance

In [ ]:
from pyspark.sql.functions import col, lit, current_timestamp, when, isnan
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import RandomForestClassificationModel

# Load the model saved by the training notebook
model = RandomForestClassificationModel.load('Files/models/maintenance_failure_rf')
print('Model loaded')

In [ ]:
# Rebuild the exact same feature set used during training so the vector matches the model
df = spark.read.table('gold_ml_features')

# Clean nulls / NaN the same way training did
for c, dtype in df.dtypes:
    if dtype in ('double', 'float'):
        df = df.withColumn(c, when(col(c).isNull() | isnan(col(c)), lit(0.0)).otherwise(col(c)))
    elif dtype in ('int', 'bigint', 'long'):
        df = df.withColumn(c, when(col(c).isNull(), lit(0)).otherwise(col(c)))

exclude = ('had_failure', 'failure_event_count', 'total_failure_downtime', 'total_failure_defects')
feature_cols = [
    c for c, dtype in df.dtypes
    if dtype in ('double', 'float', 'int', 'bigint', 'long') and c not in exclude
]
print(f'Features ({len(feature_cols)}): {feature_cols}')

assembler = VectorAssembler(inputCols=feature_cols, outputCol='features', handleInvalid='keep')
model_df = assembler.transform(df).select('features', col('had_failure').cast('double').alias('label'))

# Same split + seed as training, so we evaluate on the held-out test set
_, test_df = model_df.randomSplit([0.8, 0.2], seed=42)
predictions = model.transform(test_df)
print(f'Test predictions: {predictions.count()} rows')

In [ ]:
# Confusion matrix + precision / recall
print('=== Confusion Matrix ===')
predictions.groupBy('label', 'prediction').count().orderBy('label', 'prediction').show()

tp = predictions.filter((col('label') == 1) & (col('prediction') == 1)).count()
fp = predictions.filter((col('label') == 0) & (col('prediction') == 1)).count()
fn = predictions.filter((col('label') == 1) & (col('prediction') == 0)).count()
tn = predictions.filter((col('label') == 0) & (col('prediction') == 0)).count()

precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
print(f'TP={tp}  FP={fp}  FN={fn}  TN={tn}')
print(f'Precision: {precision:.4f}')
print(f'Recall:    {recall:.4f}')

In [ ]:
# Feature importance from the RandomForest model (Spark stores it as a sparse vector)
importances = model.featureImportances.toArray()
rows = sorted(zip(feature_cols, [float(x) for x in importances]), key=lambda r: r[1], reverse=True)

print('=== Top 10 Features ===')
for name, imp in rows[:10]:
    print(f'  {name:30s} {imp:.4f}')

fi_spark = (
    spark.createDataFrame(rows, ['feature', 'importance'])
    .withColumn('model_timestamp', current_timestamp())
)
fi_spark.write.mode('overwrite').option('overwriteSchema', 'true').format('delta').saveAsTable('gold_ml_feature_importance')
print('Feature importance saved to gold_ml_feature_importance')

In [ ]:
spark.sql('OPTIMIZE gold_ml_feature_importance')
print('Evaluation complete')